In [7]:
import pandas as pd
from pathlib import Path


In [8]:
PATH = Path("../data/raw/")

IF = pd.read_csv(PATH / "IF_1872_2026" / "results.csv")

IF.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [16]:
print(IF.shape)
print("---" * 30)
print(IF.info())
print("---" * 30)
print(IF.columns)
print("---" * 30)
print(IF.isna().sum().sort_values(ascending=False))

(49477, 9)
------------------------------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49433 non-null  float64
 4   away_score  49433 non-null  float64
 5   tournament  49477 non-null  str    
 6   city        49477 non-null  str    
 7   country     49477 non-null  str    
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB
None
------------------------------------------------------------------------------------------
Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral'],
      dtype='str')
---------------------------------------------------------

In [20]:
IF["date"] = pd.to_datetime(IF["date"])

print(IF["date"].min())
print(IF["date"].max())

IF["tournament"].value_counts().head(20)

1872-11-30 00:00:00
2026-06-27 00:00:00


tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64

In [18]:
IF[IF["home_score"].isna()][
    ["date", "home_team", "away_team", "tournament"]
].head(20)

,date,home_team,away_team,tournament
49433,2026-06-19,Scotland,Morocco,FIFA World Cup
49434,2026-06-19,Brazil,Haiti,FIFA World Cup
49435,2026-06-19,United States,Australia,FIFA World Cup
49436,2026-06-19,Turkey,Paraguay,FIFA World Cup
49437,2026-06-20,Germany,Ivory Coast,FIFA World Cup
49438,2026-06-20,Ecuador,Curaçao,FIFA World Cup
49439,2026-06-20,Netherlands,Sweden,FIFA World Cup
49440,2026-06-20,Tunisia,Japan,FIFA World Cup
49441,2026-06-21,Belgium,Iran,FIFA World Cup
49442,2026-06-21,New Zealand,Egypt,FIFA World Cup


Authoritative historical source:
results.csv

Strengths:
- 49,477 historical international matches
- Covers 1872–2026
- Includes scores, tournament, location, and neutral-site information
- Contains future 2026 World Cup fixtures

Key observations:
- 44 rows with missing scores correspond to future World Cup 2026 fixtures
- FIFA World Cup matches represent only a small portion of the dataset (~2%)
- Dataset contains many competition types
- neutral is already stored as boolean
- scores are float because of missing values in future fixtures

Risks:
- Team-name standardization will be required
- Future fixtures must be separated from historical training data
- Very old matches may not be representative of modern football

In [22]:
len(set(IF["home_team"]).union(set(IF["away_team"])))

336

In [24]:
sorted(list(set(IF["home_team"]).union(set(IF["away_team"]))))[:50]

['Abkhazia',
 'Afghanistan',
 'Albania',
 'Alderney',
 'Algeria',
 'Ambazonia',
 'American Samoa',
 'Andalusia',
 'Andorra',
 'Angola',
 'Anguilla',
 'Antigua and Barbuda',
 'Arameans Suryoye',
 'Argentina',
 'Armenia',
 'Artsakh',
 'Aruba',
 'Asturias',
 'Australia',
 'Austria',
 'Aymara',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Barawa',
 'Barbados',
 'Basque Country',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bermuda',
 'Bhutan',
 'Biafra',
 'Bolivia',
 'Bonaire',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'British Virgin Islands',
 'Brittany',
 'Brunei',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Canary Islands']

results.csv is not a FIFA-only dataset.

It's a dataset of:

International football results

which includes:

FIFA nations
historical nations
regional teams
unofficial teams
breakaway regions
cultural teams

Next step is too filter only to the corresponding teams/countries that are playing in the World Cup

In [26]:
wc2026_teams = pd.read_csv(PATH / "FIFA_WC_1930_2026" / "wc_2026_teams.csv")

wc2026_teams.head()

,team,group,confederation,fifa_rank,coach,best_wc_result,debut_2026
0,Mexico,A,CONCACAF,15,Javier Aguirre,"Quarter-finals (1970,1986)",No
1,South Africa,A,CAF,60,Hugo Broos,"Group stage (1998,2002,2010)",No
2,South Korea,A,AFC,25,Hong Myung-bo,Semi-finals (2002),No
3,Czechia,A,UEFA,41,Ivan Hasek,"Runner-up (1934,1962)",No
4,Canada,B,CONCACAF,30,Jesse Marsch,"Group stage (1986,2022)",No


"Can my historical dataset and my 2026 dataset talk to each other?"

In [ ]:
results_teams = set(IF["home_team"]).union(set(IF["away_team"]))
wc_teams = set(wc2026_teams["team"])

print(len(wc_teams))

missing = wc_teams - results_teams
extra = results_teams - wc_teams

print(missing)
print(extra)

48
{'Türkiye', 'Czechia', 'USA'}
{'Taiwan', 'Armenia', 'Pakistan', 'Malaysia', 'Madagascar', 'Guyana', 'Benin', 'Shetland', 'Tajikistan', 'Guinea-Bissau', 'Wales', 'Poland', 'Chad', 'Felvidék', 'Jersey', 'Yemen DPR', 'United States Virgin Islands', 'Western Armenia', 'Kárpátalja', 'Galicia', 'Yorkshire', 'Chechnya', 'Catalonia', 'Guernsey', 'Yugoslavia', 'Raetia', 'Andorra', 'Papua New Guinea', 'Saugeais', 'Greece', 'Gozo', 'Costa Rica', 'Basque Country', 'Zimbabwe', 'Cook Islands', 'Lebanon', 'Bolivia', 'Liberia', 'Saint Barthélemy', 'Kabylia', 'Aymara', 'Saint Martin', 'French Guiana', 'Afghanistan', 'South Sudan', 'Isle of Wight', 'Bermuda', 'South Yemen', 'Corsica', 'Surrey', 'Gambia', 'North Macedonia', 'Gibraltar', 'Botswana', 'Guadeloupe', 'Turks and Caicos Islands', 'Orkney', 'Cascadia', 'Aruba', 'Singapore', 'Bonaire', 'Malawi', 'Czech Republic', 'Bahamas', 'Hungary', 'Myanmar', 'Brunei', 'Falkland Islands', 'Artsakh', 'Seychelles', 'Faroe Islands', 'Maule Sur', 'Montserrat', 

Finding 9

All 48 World Cup 2026 teams exist in the historical results dataset.

Only three naming mismatches were identified:

- Türkiye ↔ Turkey
- Czechia ↔ Czech Republic
- USA ↔ United States

This indicates that team-name standardization will be required before performing joins between the historical results dataset and the World Cup 2026 reference datasets.

In [31]:
print(wc2026_teams.info())
print("---" * 30)
print(wc2026_teams.head())
print("---" * 30)
print(wc2026_teams.isna().sum())
print("---" * 30)
print(wc2026_teams["confederation"].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   team            48 non-null     str  
 1   group           48 non-null     str  
 2   confederation   48 non-null     str  
 3   fifa_rank       48 non-null     int64
 4   coach           48 non-null     str  
 5   best_wc_result  48 non-null     str  
 6   debut_2026      48 non-null     str  
dtypes: int64(1), str(6)
memory usage: 2.8 KB
None
------------------------------------------------------------------------------------------
           team group confederation  fifa_rank           coach  \
0        Mexico     A      CONCACAF         15  Javier Aguirre   
1  South Africa     A           CAF         60      Hugo Broos   
2   South Korea     A           AFC         25   Hong Myung-bo   
3       Czechia     A          UEFA         41      Ivan Hasek   
4        Canada     B      CONCACAF         30    

Finding 10

wc_2026_teams.csv is a high-quality reference dataset.

The dataset contains:
- 48 qualified teams
- no missing values
- no duplicate teams
- team metadata including group, confederation, FIFA ranking, coach, historical World Cup performance, and debut status.

The team column appears suitable as the primary identifier.

This dataset is the strongest candidate to become the project's authoritative team reference table.

In [32]:
wc2026_teams["team"].nunique()

48

In [11]:
elo_ratings = pd.read_csv(PATH / "FIFA_WK_Elo_Ratings" / "elo_ratings_wc2026.csv")

elo_ratings.head()

,year,snapshot_date,country,rank,country_code,rating,rank_max,rating_max,rank_avg,rating_avg,...,matches_home,matches_away,matches_neutral,wins,losses,draws,goals_for,goals_against,confederation,is_host
0,1901,1901-12-31,England,1,EN,2013,1,2079,2,1989,...,38,35,0,46,16,11,262,102,UEFA,0
1,1901,1901-12-31,Scotland,2,SQ,1973,1,2104,1,2018,...,37,37,0,53,9,12,277,101,UEFA,0
2,1902,1902-12-31,Argentina,1,AR,2021,1,2021,1,2021,...,0,1,0,1,0,0,6,0,CONMEBOL,0
3,1902,1902-12-31,England,2,EN,1995,1,2079,2,1989,...,39,38,0,47,16,14,266,105,UEFA,0
4,1902,1902-12-31,Scotland,3,SQ,1983,1,2104,1,2017,...,39,40,0,56,9,14,293,106,UEFA,0


In [35]:
print(elo_ratings.shape)
print("---" * 30)
print(elo_ratings.info())
print("---" * 30)
print(elo_ratings.isna().sum())
print("---" * 30)
print(elo_ratings["country"].nunique())
print("---" * 30)
set(wc2026_teams["team"]) - set(elo_ratings["country"])

(4683, 23)
------------------------------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 4683 entries, 0 to 4682
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   year             4683 non-null   int64
 1   snapshot_date    4683 non-null   str  
 2   country          4683 non-null   str  
 3   rank             4683 non-null   int64
 4   country_code     4683 non-null   str  
 5   rating           4683 non-null   int64
 6   rank_max         4683 non-null   int64
 7   rating_max       4683 non-null   int64
 8   rank_avg         4683 non-null   int64
 9   rating_avg       4683 non-null   int64
 10  rank_min         4683 non-null   int64
 11  rating_min       4683 non-null   int64
 12  matches_total    4683 non-null   int64
 13  matches_home     4683 non-null   int64
 14  matches_away     4683 non-null   int64
 15  matches_neutral  4683 non-null   int64
 16  wins 

{'Türkiye', 'USA'}

Finding 11

The Elo dataset is highly compatible with the 2026 team reference dataset.

Only two naming mismatches were identified:

- Türkiye ↔ Turkey
- USA ↔ United States

This confirms that a small team-name standardization layer will be sufficient for joining Elo ratings to team metadata.

In [38]:
elo_ratings["snapshot_date"] = pd.to_datetime(
    elo_ratings["snapshot_date"]
)

print(elo_ratings["snapshot_date"].min())
print(elo_ratings["snapshot_date"].max())

elo_ratings.groupby("country")["snapshot_date"].count().describe()

1901-12-31 00:00:00
2026-12-31 00:00:00


count     48.000000
mean      97.562500
std       24.530403
min       36.000000
25%       83.000000
50%      104.500000
75%      120.000000
max      127.000000
Name: snapshot_date, dtype: float64

In [39]:
elo_ratings.sort_values(
    ["country", "snapshot_date"]
).head(20)

,year,snapshot_date,country,rank,country_code,rating,rank_max,rating_max,rank_avg,rating_avg,...,matches_home,matches_away,matches_neutral,wins,losses,draws,goals_for,goals_against,confederation,is_host
546,1932,1932-12-31,Algeria,22,DZ,1690,22,1690,22,1690,...,0,1,0,0,1,0,0,1,CAF,0
576,1933,1933-12-31,Algeria,25,DZ,1664,21,1690,22,1687,...,1,1,0,0,2,0,0,4,CAF,0
607,1934,1934-12-31,Algeria,25,DZ,1660,21,1690,23,1676,...,2,1,0,0,2,1,2,6,CAF,0
642,1935,1935-12-31,Algeria,31,DZ,1640,21,1690,25,1670,...,3,1,0,0,3,1,4,10,CAF,0
670,1936,1936-12-31,Algeria,30,DZ,1640,21,1690,26,1663,...,3,1,0,0,3,1,4,10,CAF,0
700,1937,1937-12-31,Algeria,29,DZ,1640,21,1690,27,1658,...,3,1,0,0,3,1,4,10,CAF,0
732,1938,1938-12-31,Algeria,33,DZ,1640,21,1690,28,1655,...,3,1,0,0,3,1,4,10,CAF,0
764,1939,1939-12-31,Algeria,35,DZ,1640,21,1690,28,1653,...,3,1,0,0,3,1,4,10,CAF,0
797,1940,1940-12-31,Algeria,36,DZ,1640,21,1690,29,1652,...,3,1,0,0,3,1,4,10,CAF,0
830,1941,1941-12-31,Algeria,36,DZ,1640,21,1690,30,1650,...,3,1,0,0,3,1,4,10,CAF,0


In [40]:
elo_ratings[
    elo_ratings["country"] == "France"
][["snapshot_date", "rating"]].tail()

,snapshot_date,rating
4444,2023-12-31,2109
4493,2024-12-31,2055
4541,2025-12-31,2062
4589,2026-12-31,2081
4637,2026-05-27,2081


Finding 12

elo_ratings_wc2026.csv contains 4,683 rows and 48 countries with no missing values.

The dataset provides historical Elo ratings, rankings, match statistics, goals, and confederation information.

Coverage spans from 1901 to 2026.

Snapshot frequency is primarily annual (December 31st snapshots) with additional 2026 World Cup-related snapshots.

The dataset is not sorted chronologically and will require explicit date ordering before temporal joins.

Only two team-name mismatches were identified relative to the 2026 team reference dataset:

- Türkiye ↔ Turkey
- USA ↔ United States

The dataset is suitable as the project's authoritative team-strength source.

In [43]:
fixtures = pd.read_csv(PATH / "FIFA_WC_1930_2026" / "wc_2026_fixtures.csv")

fixtures.head()

,group,stage,team1,team2,venue,city,country,date,kickoff_et,team1_confederation,team1_fifa_rank,team1_coach,team2_confederation,team2_fifa_rank,team2_coach
0,A,Group Stage,Mexico,South Africa,Estadio Azteca,Mexico City,Mexico,2026-06-11,20:00 ET,CONCACAF,15.0,Javier Aguirre,CAF,60.0,Hugo Broos
1,A,Group Stage,South Korea,Czechia,Estadio Akron,Guadalajara,Mexico,2026-06-11,22:00 ET,AFC,25.0,Hong Myung-bo,UEFA,41.0,Ivan Hasek
2,A,Group Stage,South Korea,Mexico,Estadio Akron,Guadalajara,Mexico,2026-06-18,21:00 ET,AFC,25.0,Hong Myung-bo,CONCACAF,15.0,Javier Aguirre
3,A,Group Stage,Czechia,South Africa,Estadio Akron,Guadalajara,Mexico,2026-06-18,18:00 ET,UEFA,41.0,Ivan Hasek,CAF,60.0,Hugo Broos
4,A,Group Stage,Czechia,Mexico,Estadio Azteca,Mexico City,Mexico,2026-06-24,21:00 ET,UEFA,41.0,Ivan Hasek,CONCACAF,15.0,Javier Aguirre


In [45]:
print(fixtures.shape)
print("---" * 30)

print(fixtures.info())
print("---" * 30)

print(fixtures.isna().sum())

print(fixtures.tail())

(104, 15)
------------------------------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   group                72 non-null     str    
 1   stage                104 non-null    str    
 2   team1                104 non-null    str    
 3   team2                104 non-null    str    
 4   venue                104 non-null    str    
 5   city                 104 non-null    str    
 6   country              104 non-null    str    
 7   date                 104 non-null    str    
 8   kickoff_et           104 non-null    str    
 9   team1_confederation  72 non-null     str    
 10  team1_fifa_rank      72 non-null     float64
 11  team1_coach          72 non-null     str    
 12  team2_confederation  72 non-null     str    
 13  team2_fifa_rank      72 non-null     float64
 14  

In [46]:
fixtures.columns

Index(['group', 'stage', 'team1', 'team2', 'venue', 'city', 'country', 'date',
       'kickoff_et', 'team1_confederation', 'team1_fifa_rank', 'team1_coach',
       'team2_confederation', 'team2_fifa_rank', 'team2_coach'],
      dtype='str')

In [48]:
print(fixtures["team1"].nunique())
print(fixtures["team2"].nunique())

80
80


In [49]:
fixtures["stage"].value_counts()

stage
Group Stage        72
Round of 32        16
Round of 16         8
Quarter-final       4
Semi-final          2
3rd Place Match     1
Final               1
Name: count, dtype: int64

In [ ]:
fixture_teams = set(fixtures["team1"]).union(set(fixtures["team2"]))

len(fixture_teams)

111

In [52]:
print(fixture_teams - wc_teams)

{'Best 3rd #7', '2D', '1E', '2F', '2K', 'R32 W3', 'R32 W7', 'Finalist 1', 'SF1', 'R32 W12', 'QF8', '1D', 'Best 3rd #2', 'R32 W9', 'Best 3rd #5', '1J', 'R32 W10', '1K', '2G', '2I', 'QF7', 'Best 3rd #8', 'Best 3rd #1', 'QF3', 'R32 W11', 'SF3', 'R32 W1', 'QF5', 'R32 W6', 'QF2', 'QF4', '3rd Place', '1C', '1H', 'SF4', '2C', '2A', 'Best 3rd #4', 'R32 W2', '2L', '1L', 'R32 W15', 'Finalist 2', '1F', '2J', 'SF2', 'QF1', '2E', '1A', 'Best 3rd #3', 'R32 W5', 'R32 W8', 'R32 W13', '1G', 'QF6', 'R32 W16', '1B', '1I', 'R32 W4', 'Best 3rd #6', 'R32 W14', '2B', '2H'}
